In [1]:
import re
import unicodedata

import pandas as pd
import torch

from model import (
    get_device,
    get_input_limit,
    get_label_mapping,
    load_saved_model,
    resolve_model_dir,
)

In [2]:
def normalize_text(value, field_name):
    if not isinstance(value, str):
        raise ValueError(f"{field_name} must be a nonempty string")
    value = re.sub(r"\s+", " ", unicodedata.normalize("NFKC", value)).strip()
    if not value or value.casefold() == "null":
        raise ValueError(f"{field_name} must be a nonempty string")
    return value

In [8]:
class _ModelPredictor:
    def __init__(self, model_dir, device):
        self.model_dir = resolve_model_dir(model_dir)
        self.device = device
        self.model, self.tokenizer = load_saved_model(self.model_dir, device=device)
        self.id_to_label = get_label_mapping(self.model.config)
        self.max_length = get_input_limit(self.tokenizer, self.model.config)

    def predict_texts(self, texts):
        self.model.eval()
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        inputs = {key: value.to(self.device) for key, value in inputs.items()}
        logits = self.model(**inputs).logits
        probabilities = torch.softmax(logits.float(), dim=-1)
        scores, indices = probabilities.max(dim=-1)

        return [
            {"category": self.id_to_label[index], "score": score}
            for index, score in zip(indices.cpu().tolist(), scores.cpu().tolist())
        ]

In [9]:
class NewsClassifier:
    def __init__(self, model_1_dir, model_2_dir, device="auto"):
        model_1_dir = resolve_model_dir(model_1_dir)
        model_2_dir = resolve_model_dir(model_2_dir)
        if model_1_dir.resolve() == model_2_dir.resolve():
            raise ValueError("Provide two different checkpoint folders")

        self.device = get_device(device)
        self.model_1 = _ModelPredictor(model_1_dir, self.device)
        self.model_2 = _ModelPredictor(model_2_dir, self.device)
        labels_1 = set(self.model_1.id_to_label.values())
        labels_2 = set(self.model_2.id_to_label.values())
        if labels_1 != labels_2:
            raise ValueError("Both models must use the same category taxonomy")

    def predict(self, headline, short_description):
        return self.predict_batch(
            [{"headline": headline, "short_description": short_description}],
            batch_size=1,
        )[0]

    @torch.inference_mode()
    def predict_batch(self, articles, batch_size=8):
        if not isinstance(batch_size, int) or batch_size < 1:
            raise ValueError("batch_size must be a positive integer")

        texts = []
        for article in articles:
            if not isinstance(article, dict) or not {"headline", "short_description"}.issubset(article):
                raise ValueError("Each article must contain headline and short_description")
            headline = normalize_text(article["headline"], "headline")
            description = normalize_text(article["short_description"], "short_description")
            texts.append(headline + " " + description)

        results = []
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            predictions_1 = self.model_1.predict_texts(batch_texts)
            predictions_2 = self.model_2.predict_texts(batch_texts)
            for prediction_1, prediction_2 in zip(predictions_1, predictions_2):
                # Compare category names: the two models may assign different IDs.
                agrees = prediction_1["category"] == prediction_2["category"]
                results.append({
                    "category": prediction_1["category"] if agrees else None,
                    "requires_human_review": not agrees,
                    "model_1": prediction_1,
                    "model_2": prediction_2,
                })
        return results

    def predict_dataframe(self, frame: pd.DataFrame, batch_size=8) -> pd.DataFrame:
        required = {"headline", "short_description"}
        if missing := required - set(frame.columns):
            raise ValueError(f"Missing columns: {sorted(missing)}")
        predictions = self.predict_batch(
            frame[["headline", "short_description"]].to_dict(orient="records"),
            batch_size=batch_size,
        )
        result = frame.copy()
        result["predicted_category"] = [prediction["category"] for prediction in predictions]
        result["requires_human_review"] = [prediction["requires_human_review"] for prediction in predictions]
        for model_name in ("model_1", "model_2"):
            result[f"{model_name}_category"] = [prediction[model_name]["category"] for prediction in predictions]
            result[f"{model_name}_score"] = [prediction[model_name]["score"] for prediction in predictions]
        return result

In [10]:
MODEL_1_DIR = "artifacts/distilbert-base/best_model"
MODEL_2_DIR = "artifacts/modernbert-base/best_model"

In [11]:
classifier = NewsClassifier(
    model_1_dir=MODEL_1_DIR,
    model_2_dir=MODEL_2_DIR,
)
print("Device:", classifier.device)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Device: cuda


In [14]:
headline = "Government announces new economic policy"
short_description = "The policy aims to support small businesses."

result = classifier.predict(
    headline=headline,
    short_description=short_description,
)
if result["requires_human_review"]:
    print("Human review required")
    print("Model 1:", result["model_1"]["category"])
    print("Model 2:", result["model_2"]["category"])
else:
    print("Automatic category:", result["category"])

Automatic category: POLITICS


In [19]:
from pathlib import Path

test_df = pd.read_csv("data/news_test.csv")

ensemble_df = classifier.predict_dataframe(
    test_df,
    batch_size=8,
)

output_dir = Path("artifacts/evaluation/ensemble")
output_dir.mkdir(parents=True, exist_ok=True)

ensemble_df.to_csv(
    output_dir / "predictions.csv",
    index=False,
)

agreed = ~ensemble_df["requires_human_review"]

correct = (
    ensemble_df["predicted_category"]
    == ensemble_df["category"]
)

total = len(ensemble_df)
n_auto = int(agreed.sum())
n_review = total - n_auto
n_correct = int((agreed & correct).sum())
n_wrong = n_auto - n_correct

coverage = n_auto / total
selective_risk = n_wrong / n_auto if n_auto else float("nan")
auto_tag_precision = n_correct / n_auto if n_auto else float("nan")
human_review_rate = n_review / total

metrics = {
    "total_articles": total,
    "auto_tagged": n_auto,
    "human_review": n_review,
    "correct_auto_tags": n_correct,
    "incorrect_auto_tags": n_wrong,
    "automation_coverage": coverage,
    "selective_risk": selective_risk,
    "auto_tag_precision": auto_tag_precision,
    "human_review_rate": human_review_rate,
}

print(f"Total articles:       {total:,}")
print(f"Auto-tagged:          {n_auto:,}")
print(f"Human review:         {n_review:,}")
print(f"Correct auto-tags:    {n_correct:,}")
print(f"Incorrect auto-tags:  {n_wrong:,}")
print(f"Automation coverage:  {coverage:.2%}")
print(f"Selective risk:       {selective_risk:.2%}")
print(f"Auto-tag precision:   {auto_tag_precision:.2%}")
print(f"Human review rate:    {human_review_rate:.2%}")

pd.DataFrame([metrics]).to_csv(
    output_dir / "metrics.csv",
    index=False,
)

Total articles:       11,425
Auto-tagged:          9,615
Human review:         1,810
Correct auto-tags:    8,415
Incorrect auto-tags:  1,200
Automation coverage:  84.16%
Selective risk:       12.48%
Auto-tag precision:   87.52%
Human review rate:    15.84%
